# Laboratorio 2 — Ejercicio 1: Modelos LSTM para Series de Tiempo

**CC3084 – Data Science · Universidad del Valle de Guatemala · Semestre II 2026**

**Integrantes:** Javier España #23361 · Angel Esquit #23221 · Roberto Barreda #23354


## 0. Imports y configuración global

In [ ]:
import os, sys, itertools, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Agregar src al path para importar módulos
SRC_PATH = os.path.join('..', 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

SEMILLA    = 42
EPOCAS     = 500
VALIDACION = 24          # últimos 24 meses de train para validación interna
TRAIN_RATIO = 0.70
SERIES = ['Total_Consistent', 'Via_Aerea']

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

print(f'PyTorch {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

---
## 1.1  Preparación de datos: conjuntos de entrenamiento y prueba

Se utiliza el mismo CSV del Laboratorio 1 (`series_de_tiempo_completas.csv`). La serie se divide en **70 % entrenamiento** y **30 % prueba**, igual que en el laboratorio anterior, para que las métricas sean comparables.

In [ ]:
# --- Carga y división ---
DATA_PATH = os.path.join('..', 'data', 'series_de_tiempo_completas.csv')
df = pd.read_csv(DATA_PATH, index_col='Fecha', parse_dates=True).asfreq('MS')

fechas = sorted(df.index.unique())
corte  = pd.Timestamp(fechas[int(len(fechas) * TRAIN_RATIO)])
print(f'Fecha de corte (70 %): {corte:%Y-%m}')

def dividir(col):
    s = df[col].interpolate(method='time').fillna(0).clip(lower=0)
    return s.loc[:corte].asfreq('MS'), s.loc[corte:].iloc[1:].asfreq('MS')

conjuntos = {c: dividir(c) for c in SERIES}

for nombre, (train, test) in conjuntos.items():
    print(f'\n{nombre}:')
    print(f'  Train: {len(train)} obs ({train.index.min():%Y-%m} -> {train.index.max():%Y-%m})')
    print(f'  Test : {len(test)} obs  ({test.index.min():%Y-%m} -> {test.index.max():%Y-%m})')

In [ ]:
# Visualización de la división train / test
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

for ax, nombre in zip(axes, SERIES):
    train, test = conjuntos[nombre]
    ax.plot(train.index, train.values, label='Entrenamiento', color='#2563eb')
    ax.plot(test.index,  test.values,  label='Prueba',        color='#dc2626')
    ax.axvline(corte, color='gray', ls='--', lw=1, label='Corte 70/30')
    ax.set_title(nombre, fontsize=13, fontweight='bold')
    ax.set_ylabel('Visitantes')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('1.1 - Division Train / Test', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

---
## 1.2  Definición de modelos LSTM y tuneo de hiperparámetros

Se definen **dos arquitecturas** y una grilla de hiperparámetros. El tuneo evalúa cada combinación sobre los **últimos 24 meses del conjunto de entrenamiento** (validación interna).

| Configuración | Capas LSTM | Dropout | Ventanas | Unidades | Learning Rates |
|---------------|------------|---------|----------|----------|----------------|
| **LstmSimple**  | 1 | 0.0 | 12, 24 | 32, 64 | 0.01, 0.001 |
| **LstmApilado** | 2 | 0.2 | 12, 24 | 32, 64 | 0.01, 0.001 |

Esto genera **8 combinaciones por arquitectura x 2 arquitecturas = 16 modelos** por serie.

In [ ]:
# --- Arquitectura LSTM ---
class RedLstm(nn.Module):
    """Red LSTM configurable: 1 o mas capas, con dropout opcional."""
    def __init__(self, unidades, capas, dropout):
        super().__init__()
        self.lstm = nn.LSTM(1, unidades, capas, batch_first=True,
                            dropout=dropout if capas > 1 else 0.0)
        self.salida = nn.Linear(unidades, 1)

    def forward(self, x):
        h, _ = self.lstm(x)
        return self.salida(h[:, -1])

# --- Funciones auxiliares ---
def crear_secuencias(valores, ventana):
    """Genera pares (X, y) con ventanas deslizantes."""
    X = np.array([valores[i:i+ventana] for i in range(len(valores)-ventana)])
    y = np.array([valores[i+ventana]   for i in range(len(valores)-ventana)])
    return (torch.tensor(X, dtype=torch.float32).unsqueeze(-1),
            torch.tensor(y, dtype=torch.float32).unsqueeze(-1))

def entrenar_modelo(x, y, unidades, capas, dropout, lr, epocas=EPOCAS):
    """Entrena una RedLstm y devuelve (modelo, perdida_final)."""
    torch.manual_seed(SEMILLA)
    modelo = RedLstm(unidades, capas, dropout)
    opt    = torch.optim.Adam(modelo.parameters(), lr=lr)
    crit   = nn.MSELoss()
    modelo.train()
    for _ in range(epocas):
        opt.zero_grad()
        loss = crit(modelo(x), y)
        loss.backward()
        opt.step()
    return modelo, float(loss.detach())

def calc_mae(real, pred):
    return float(np.mean(np.abs(real - pred)))

def calc_rmse(real, pred):
    return float(np.sqrt(np.mean((real - pred)**2)))

print('Arquitectura y funciones definidas OK')

In [ ]:
# --- Grilla de hiperparámetros ---
CONFIGURACIONES = {
    'LstmSimple':  {'capas': [1], 'dropout': [0.0],
                    'ventana': [12, 24], 'unidades': [32, 64], 'lr': [0.01, 0.001]},
    'LstmApilado': {'capas': [2], 'dropout': [0.2],
                    'ventana': [12, 24], 'unidades': [32, 64], 'lr': [0.01, 0.001]},
}

def gen_combinaciones(rejilla):
    claves = list(rejilla)
    return [dict(zip(claves, v)) for v in itertools.product(*rejilla.values())]

total_combos = sum(len(gen_combinaciones(r)) for r in CONFIGURACIONES.values())
print(f'Total de combinaciones por serie: {total_combos}')

In [ ]:
# --- Ejecución del tuneo ---
def tunear_serie(train):
    """Evalua todas las combinaciones usando los ultimos VALIDACION meses de train."""
    ajuste    = train.iloc[:-VALIDACION]
    escalador = MinMaxScaler().fit(ajuste.values.reshape(-1, 1))
    escalado  = escalador.transform(train.values.reshape(-1, 1)).ravel()
    real_val  = train.values[-VALIDACION:]

    filas = []
    for nombre_cfg, rejilla in CONFIGURACIONES.items():
        for p in gen_combinaciones(rejilla):
            X, Y = crear_secuencias(escalado, p['ventana'])
            corte_val = len(Y) - VALIDACION
            modelo, perdida = entrenar_modelo(
                X[:corte_val], Y[:corte_val],
                p['unidades'], p['capas'], p['dropout'], p['lr'])
            modelo.eval()
            with torch.no_grad():
                pred_esc = modelo(X[corte_val:]).numpy().reshape(-1, 1)
            pred = escalador.inverse_transform(pred_esc).ravel()
            filas.append({
                'Configuracion': nombre_cfg, **p,
                'PerdidaEntrenamiento': round(perdida, 6),
                'ValMae':  round(calc_mae(real_val, pred), 2),
                'ValRmse': round(calc_rmse(real_val, pred), 2),
            })
    return pd.DataFrame(filas)

print('Ejecutando tuneo (16 modelos x 2 series = 32 entrenamientos)...')
print('Esto puede tardar unos minutos.\n')

tablas_tuneo = {}
for nombre in SERIES:
    train, _ = conjuntos[nombre]
    tabla = tunear_serie(train)
    tabla.insert(0, 'Serie', nombre)
    tablas_tuneo[nombre] = tabla
    print(f'OK Tuneo completado para {nombre}')

tuneo_df = pd.concat(tablas_tuneo.values(), ignore_index=True)
os.makedirs(os.path.join('..', 'results'), exist_ok=True)
tuneo_df.to_csv(os.path.join('..', 'results', 'TuneoLstm.csv'), index=False)
print('\nResultados guardados en results/TuneoLstm.csv')

In [ ]:
# --- Tabla de tuneo para Total_Consistent (ordenada por ValRmse) ---
print('=== Total_Consistent - Resultados de Tuneo ===')
display(tablas_tuneo['Total_Consistent']
        .drop(columns='Serie')
        .sort_values('ValRmse')
        .reset_index(drop=True)
        .style.highlight_min(subset=['ValMae', 'ValRmse'], color='#c6efce'))

In [ ]:
# --- Tabla de tuneo para Via_Aerea (ordenada por ValRmse) ---
print('=== Via_Aerea - Resultados de Tuneo ===')
display(tablas_tuneo['Via_Aerea']
        .drop(columns='Serie')
        .sort_values('ValRmse')
        .reset_index(drop=True)
        .style.highlight_min(subset=['ValMae', 'ValRmse'], color='#c6efce'))

In [ ]:
# --- Selección del mejor modelo global por serie ---
mejor_idx = tuneo_df.groupby('Serie')['ValRmse'].idxmin()
mejores   = tuneo_df.loc[mejor_idx].reset_index(drop=True)
mejores.to_csv(os.path.join('..', 'results', 'MejoresLstm.csv'), index=False)

print('=================================================================')
print('  MEJOR MODELO SELECCIONADO POR SERIE (menor ValRmse)')
print('=================================================================')
display(mejores)

In [ ]:
# --- Visualización del tuneo: RMSE de validación por configuración y serie ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, nombre in zip(axes, SERIES):
    df_plot = tablas_tuneo[nombre].copy()
    df_plot['Etiqueta'] = (df_plot['Configuracion'] + '\nv' +
                           df_plot['ventana'].astype(str) + '-u' +
                           df_plot['unidades'].astype(str) + '-lr' +
                           df_plot['lr'].astype(str))
    colores = ['#2563eb' if c == 'LstmSimple' else '#dc2626'
               for c in df_plot['Configuracion']]
    bars = ax.barh(df_plot['Etiqueta'], df_plot['ValRmse'], color=colores, alpha=0.8)
    ax.set_xlabel('RMSE (Validacion)', fontsize=11)
    ax.set_title(f'{nombre}\nTuneo de Hiperparametros', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    p1 = mpatches.Patch(color='#2563eb', label='LstmSimple (1 capa)')
    p2 = mpatches.Patch(color='#dc2626', label='LstmApilado (2 capas)')
    ax.legend(handles=[p1, p2], loc='lower right', fontsize=9)

fig.suptitle('1.2 - RMSE de Validacion por Configuracion LSTM', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.savefig(os.path.join('..', 'results', 'tuneo_barras.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 1.3  Predicción sobre el conjunto de prueba con el mejor modelo

Se re-entrena cada mejor modelo usando **todo el conjunto de entrenamiento** (sin reservar validación) y se genera la predicción sobre el conjunto de prueba.

Se emplean dos estrategias de predicción:

- **Multi-step (Autoregresivo):** el modelo se alimenta recursivamente de sus propias predicciones para generar todo el horizonte de prueba.
- **One-step ahead:** se utiliza el valor real del paso anterior para predecir el siguiente.

In [ ]:
def predecir_test(train, test, params, autoregresivo=True):
    """
    Re-entrena el mejor modelo con TODO el train y predice sobre test.
    autoregresivo=True  -> Multi-step: prediccion recursiva
    autoregresivo=False -> One-step ahead: usa valores reales como contexto
    """
    torch.manual_seed(SEMILLA)
    escalador = MinMaxScaler().fit(train.values.reshape(-1, 1))
    train_esc = escalador.transform(train.values.reshape(-1, 1)).ravel()
    test_esc  = escalador.transform(test.values.reshape(-1, 1)).ravel()

    ventana = params['ventana']
    X_tr, Y_tr = crear_secuencias(train_esc, ventana)
    modelo, _ = entrenar_modelo(X_tr, Y_tr,
                                params['unidades'], params['capas'],
                                params['dropout'],  params['lr'])
    modelo.eval()

    if autoregresivo:
        historia = list(train_esc[-ventana:])
        preds_esc = []
        for _ in range(len(test)):
            inp = torch.tensor(historia[-ventana:],
                               dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
            with torch.no_grad():
                out = modelo(inp).item()
            preds_esc.append(out)
            historia.append(out)
        preds = escalador.inverse_transform(
            np.array(preds_esc).reshape(-1, 1)).ravel()
    else:
        full_esc = np.concatenate([train_esc[-ventana:], test_esc])
        X_te = np.array([full_esc[i:i+ventana] for i in range(len(test))])
        inp = torch.tensor(X_te, dtype=torch.float32).unsqueeze(-1)
        with torch.no_grad():
            preds_esc = modelo(inp).numpy().reshape(-1, 1)
        preds = escalador.inverse_transform(preds_esc).ravel()

    return preds

print('Funcion de prediccion definida OK')

In [ ]:
# --- Ejecutar predicciones con el mejor modelo de cada serie ---
resultados_test = []
predicciones = {}

for _, row in mejores.iterrows():
    serie = row['Serie']
    train, test = conjuntos[serie]
    params = {
        'unidades': int(row['unidades']),
        'capas':    int(row['capas']),
        'dropout':  float(row['dropout']),
        'ventana':  int(row['ventana']),
        'lr':       float(row['lr']),
    }

    print(f'\nPrediciendo {serie} con: {row["Configuracion"]}, '
          f'ventana={params["ventana"]}, unidades={params["unidades"]}, lr={params["lr"]}...')

    preds_ar = predecir_test(train, test, params, autoregresivo=True)
    preds_os = predecir_test(train, test, params, autoregresivo=False)

    mae_ar  = calc_mae(test.values,  preds_ar)
    rmse_ar = calc_rmse(test.values, preds_ar)
    mae_os  = calc_mae(test.values,  preds_os)
    rmse_os = calc_rmse(test.values, preds_os)

    resultados_test.append({
        'Serie': serie,
        'Configuracion': row['Configuracion'],
        'Ventana': params['ventana'],
        'Unidades': params['unidades'],
        'LR': params['lr'],
        'Test_MAE_Multistep':  round(mae_ar, 2),
        'Test_RMSE_Multistep': round(rmse_ar, 2),
        'Test_MAE_OneStep':    round(mae_os, 2),
        'Test_RMSE_OneStep':   round(rmse_os, 2),
    })
    predicciones[serie] = (preds_ar, preds_os)

    pd.DataFrame({
        'Fecha': test.index,
        'Real': test.values,
        'Pred_Multistep': preds_ar,
        'Pred_OneStep':   preds_os,
    }).to_csv(os.path.join('..', 'results', f'predicciones_{serie.lower()}.csv'), index=False)

    print(f'  Multi-step ->  MAE: {mae_ar:,.2f}  |  RMSE: {rmse_ar:,.2f}')
    print(f'  One-step   ->  MAE: {mae_os:,.2f}  |  RMSE: {rmse_os:,.2f}')

df_test = pd.DataFrame(resultados_test)
df_test.to_csv(os.path.join('..', 'results', 'ResultadosTestLSTM.csv'), index=False)
print('\nOK Predicciones guardadas en results/')

In [ ]:
# --- Tabla resumen de resultados en test ---
print('=======================================================================')
print('  RESULTADOS EN CONJUNTO DE PRUEBA - MEJOR MODELO LSTM POR SERIE')
print('=======================================================================')
display(df_test.style.format({
    'Test_MAE_Multistep':  '{:,.2f}',
    'Test_RMSE_Multistep': '{:,.2f}',
    'Test_MAE_OneStep':    '{:,.2f}',
    'Test_RMSE_OneStep':   '{:,.2f}',
}))

In [ ]:
# --- Graficas de prediccion ---
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

for ax, serie in zip(axes, SERIES):
    train, test = conjuntos[serie]
    preds_ar, preds_os = predicciones[serie]

    ax.plot(train.index[-36:], train.values[-36:],
            color='#475569', lw=1.5, alpha=0.7, label='Train (ultimos 3 anos)')
    ax.plot(test.index, test.values,
            color='#2563eb', lw=2, label='Test real')
    ax.plot(test.index, preds_ar,
            color='#dc2626', lw=1.8, ls='--', label='LSTM Multi-step')
    ax.plot(test.index, preds_os,
            color='#16a34a', lw=1.8, ls=':', label='LSTM One-step')
    ax.axvline(test.index[0], color='gray', ls='--', lw=0.8)

    ax.set_title(f'Predicciones LSTM - {serie}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Visitantes')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

fig.suptitle('1.3 - Prediccion con el mejor modelo LSTM', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
plt.savefig(os.path.join('..', 'results', 'predicciones_lstm.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 1.4  Analisis comparativo: cuál predijo mejor? Son mejores que los modelos del Lab 1?

En esta seccion se responden las tres preguntas del enunciado:

1. Cual de las dos series fue predicha mejor por el LSTM?
2. Son los modelos LSTM mejores que los del Laboratorio 1?
3. Como se determino? — Evidencia con metricas (MAE, RMSE).

In [ ]:
# Cargar metricas del Laboratorio 1
LAB1_PATH = os.path.join('..', 'data', 'lab1_model_comparison_metrics.csv')
lab1 = pd.read_csv(LAB1_PATH)

lab1_filtrado = lab1[lab1['Serie'].isin(SERIES)].copy()

idx_mejor_lab1 = lab1_filtrado.groupby('Serie')['RMSE'].idxmin()
lab1_mejor = lab1_filtrado.loc[idx_mejor_lab1].reset_index(drop=True)

print('===========================================')
print('  MEJOR MODELO LABORATORIO 1 (por RMSE)')
print('===========================================')
display(lab1_mejor[['Serie', 'Modelo', 'MAE', 'RMSE']])

In [ ]:
# Construir tabla unificada para comparacion
filas_comparacion = []

for _, row_lstm in df_test.iterrows():
    serie = row_lstm['Serie']

    lstm_mae_ms  = row_lstm['Test_MAE_Multistep']
    lstm_rmse_ms = row_lstm['Test_RMSE_Multistep']
    lstm_mae_os  = row_lstm['Test_MAE_OneStep']
    lstm_rmse_os = row_lstm['Test_RMSE_OneStep']

    mejor_lab1 = lab1_mejor[lab1_mejor['Serie'] == serie].iloc[0]

    mejora_ms = round((mejor_lab1['RMSE'] - lstm_rmse_ms) / mejor_lab1['RMSE'] * 100, 1)
    mejora_os = round((mejor_lab1['RMSE'] - lstm_rmse_os) / mejor_lab1['RMSE'] * 100, 1)

    filas_comparacion.append({
        'Serie': serie,
        'Lab1_Modelo':      mejor_lab1['Modelo'],
        'Lab1_MAE':         mejor_lab1['MAE'],
        'Lab1_RMSE':        mejor_lab1['RMSE'],
        'LSTM_Conf':        row_lstm['Configuracion'],
        'LSTM_MAE_MS':      lstm_mae_ms,
        'LSTM_RMSE_MS':     lstm_rmse_ms,
        'LSTM_MAE_OS':      lstm_mae_os,
        'LSTM_RMSE_OS':     lstm_rmse_os,
        'MejoraMSvLab1_%':  mejora_ms,
        'MejoraOSvLab1_%':  mejora_os,
    })

comp_df = pd.DataFrame(filas_comparacion)
comp_df.to_csv(os.path.join('..', 'results', 'comparacion_lstm_vs_lab1.csv'), index=False)

print('===========================================================================')
print('  COMPARACION LSTM vs. MEJOR MODELO LAB 1')
print('===========================================================================')
display(comp_df.style.format({
    'Lab1_MAE':      '{:,.2f}',
    'Lab1_RMSE':     '{:,.2f}',
    'LSTM_MAE_MS':   '{:,.2f}',
    'LSTM_RMSE_MS':  '{:,.2f}',
    'LSTM_MAE_OS':   '{:,.2f}',
    'LSTM_RMSE_OS':  '{:,.2f}',
    'MejoraMSvLab1_%': '{:+.1f}%',
    'MejoraOSvLab1_%': '{:+.1f}%',
}).background_gradient(subset=['MejoraMSvLab1_%', 'MejoraOSvLab1_%'], cmap='RdYlGn'))

In [ ]:
# Grafica comparativa: RMSE de todos los modelos (Lab1 + LSTM) por serie
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=False)

colores_lab1 = {
    'SARIMA':                  '#6366f1',
    'Seasonal Naive':          '#94a3b8',
    'Suav. Exponencial (SES)': '#f59e0b',
    'Holt-Winters':            '#10b981',
    'Prophet':                 '#8b5cf6',
}

for ax, serie in zip(axes, SERIES):
    d1 = lab1_filtrado[lab1_filtrado['Serie'] == serie].copy()
    lstm_row = df_test[df_test['Serie'] == serie].iloc[0]
    lstm_ms_rmse = lstm_row['Test_RMSE_Multistep']
    lstm_os_rmse = lstm_row['Test_RMSE_OneStep']

    nombres = list(d1['Modelo']) + ['LSTM Multi-step', 'LSTM One-step']
    valores  = list(d1['RMSE'].values) + [lstm_ms_rmse, lstm_os_rmse]
    colores  = [colores_lab1.get(m, '#64748b') for m in d1['Modelo']] + ['#dc2626', '#16a34a']

    bars = ax.bar(nombres, valores, color=colores, alpha=0.85, edgecolor='white', linewidth=0.8)
    ax.set_title(f'{serie}\nComparacion RMSE en Test', fontsize=12, fontweight='bold')
    ax.set_ylabel('RMSE (Visitantes)')
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(valores)*0.01,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    # Resaltar el mejor modelo con borde dorado
    min_idx = valores.index(min(valores))
    bars[min_idx].set_edgecolor('#f59e0b')
    bars[min_idx].set_linewidth(3)

fig.suptitle('1.4 - RMSE en Conjunto de Prueba: Lab 1 vs. LSTM\n(borde dorado = mejor modelo global)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig(os.path.join('..', 'results', 'comparacion_rmse_lab1_vs_lstm.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# NRMSE: RMSE relativo normalizado por la media del test para comparacion justa entre series
print('=================================================================')
print('  RMSE RELATIVO (RMSE / Media Test) — menor es mejor')
print('=================================================================')

tabla_relativa = []
for _, row in df_test.iterrows():
    serie = row['Serie']
    _, test = conjuntos[serie]
    media_test = test.values.mean()
    tabla_relativa.append({
        'Serie':            serie,
        'Media Test':       round(media_test, 0),
        'RMSE Multi-step':  row['Test_RMSE_Multistep'],
        'RMSE One-step':    row['Test_RMSE_OneStep'],
        'NRMSE Multi-step': round(row['Test_RMSE_Multistep'] / media_test * 100, 2),
        'NRMSE One-step':   round(row['Test_RMSE_OneStep']   / media_test * 100, 2),
    })

df_relativa = pd.DataFrame(tabla_relativa)
display(df_relativa.style.highlight_min(subset=['NRMSE Multi-step', 'NRMSE One-step'],
                                         color='#c6efce'))

mejor_serie_ms = df_relativa.loc[df_relativa['NRMSE Multi-step'].idxmin(), 'Serie']
mejor_serie_os = df_relativa.loc[df_relativa['NRMSE One-step'].idxmin(), 'Serie']
print(f'\n>> Serie predicha mejor (Multi-step): {mejor_serie_ms}')
print(f'>> Serie predicha mejor (One-step):   {mejor_serie_os}')

In [ ]:
# Grafica de error absoluto en el tiempo por serie
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

for ax, serie in zip(axes, SERIES):
    _, test = conjuntos[serie]
    preds_ar, preds_os = predicciones[serie]

    err_ms = np.abs(test.values - preds_ar)
    err_os = np.abs(test.values - preds_os)

    ax.fill_between(test.index, err_ms, alpha=0.25, color='#dc2626')
    ax.fill_between(test.index, err_os, alpha=0.25, color='#16a34a')
    ax.plot(test.index, err_ms, color='#dc2626', lw=1.5, label='Error Multi-step')
    ax.plot(test.index, err_os, color='#16a34a', lw=1.5, label='Error One-step')

    ax.set_title(f'Error Absoluto en Test - {serie}', fontsize=12, fontweight='bold')
    ax.set_ylabel('|Real - Predicho|')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('1.4 - Error Absoluto en Tiempo (LSTM Multi-step vs. One-step)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig(os.path.join('..', 'results', 'error_absoluto_lstm.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tabla final: veredicto LSTM vs Lab 1
print('=======================================================================')
print('  VEREDICTO FINAL - LSTM supera a Lab 1?')
print('=======================================================================')

veredicto_filas = []
for _, row in comp_df.iterrows():
    mejora_os = row['MejoraOSvLab1_%']
    mejora_ms = row['MejoraMSvLab1_%']
    veredicto_os = 'LSTM mejor (OS)' if mejora_os > 0 else 'Lab1 mejor (OS)'
    veredicto_ms = 'LSTM mejor (MS)' if mejora_ms > 0 else 'Lab1 mejor (MS)'
    veredicto_filas.append({
        'Serie':              row['Serie'],
        'Mejor Lab1':         row['Lab1_Modelo'],
        'RMSE Lab1':          f"{row['Lab1_RMSE']:,.0f}",
        'Mejor LSTM':         row['LSTM_Conf'],
        'RMSE LSTM MS':       f"{row['LSTM_RMSE_MS']:,.0f}",
        'RMSE LSTM OS':       f"{row['LSTM_RMSE_OS']:,.0f}",
        'Mejora MS %':        f"{mejora_ms:+.1f}%",
        'Mejora OS %':        f"{mejora_os:+.1f}%",
        'Veredicto':          veredicto_os,
    })

df_veredicto = pd.DataFrame(veredicto_filas)
display(df_veredicto)

### 1.4.1  Cual serie fue predicha mejor?

**Criterio:** RMSE en el conjunto de prueba como metrica principal, complementado con NRMSE (RMSE normalizado por la media de la serie) para comparacion justa entre series de diferentes escalas.

- **Via_Aerea** suele presentar un **NRMSE mas bajo**, lo que indica que el LSTM captura mejor el patron relativo de esta serie. Esto se debe a que Via_Aerea tiene menor varianza absoluta y un patron estacional mas regular.
- **Total_Consistent** contiene el impacto de la pandemia (2020-2021) de manera mas aguda en terminos absolutos, con un quiebre estructural que incrementa el error.

### 1.4.2  Son mejores que los modelos del Laboratorio 1?

**Criterio de comparacion:**
- Se compara el RMSE del mejor modelo LSTM contra el mejor modelo de Lab 1 (Holt-Winters en ambas series).
- La mejora se expresa como: Mejora% = (RMSE_Lab1 - RMSE_LSTM) / RMSE_Lab1 * 100
- Valores positivos: LSTM supera al modelo clasico. Negativos: el modelo clasico es mejor.

| Aspecto | Observacion |
|---------|-------------|
| **One-step ahead** | Comparable directamente con modelos clasicos. Ambos usan valores reales previos. |
| **Multi-step** | Escenario real de prediccion a futuro. Acumula error por retroalimentacion. |
| **Holt-Winters (Lab1)** | Mejor modelo clasico en ambas series. Captura bien la estacionalidad. |
| **LSTM ventaja** | Puede capturar no-linealidades que los modelos clasicos no modelan. |
| **LSTM desventaja** | Requiere mas datos, mas tiempo de entrenamiento y es mas sensible al quiebre pandemia. |

### 1.4.3  Como se determino?

1. **MAE (Error Absoluto Medio):** Error promedio en unidades de visitantes. Robusto a outliers.
2. **RMSE (Raiz del Error Cuadratico Medio):** Penaliza errores grandes. Criterio principal de seleccion.
3. **NRMSE:** RMSE normalizado, permite comparacion entre series de diferente escala.
4. **Porcentaje de mejora:** Cuantifica directamente la superioridad del LSTM vs modelo clasico.

**Nota sobre el periodo 2019-2021:** Los meses del desplome pandemia concentran casi todo el error tanto en modelos clasicos como en LSTM. Ningun modelo anticipo este quiebre estructural en entrenamiento. Sin embargo, el LSTM puede recuperarse mas rapidamente despues del quiebre gracias a su capacidad de capturar dependencias no lineales en los datos post-pandemia disponibles en el conjunto de prueba.
